# cAPTure: Stage-2 graph-input audit

This notebook validates the boundary between the immutable FULL_DEV graph materialization and Stage-2 model code. It stages the compressed shards locally, verifies every checksum, reconstructs PyTorch Geometric graphs, and checks fold roles, chronological order, time gaps, packet-edge correspondence, and the exact global-node-ID-to-MAC mapping.

It does not train models and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Mount Drive and prepare the graph environment

Push the current branch before opening Colab. This installs PyTorch and PyTorch Geometric for graph reconstruction only; no optimization is performed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_IDENTITY_WORK_ROOT = Path("/content/capture_graph_identity_work")

REPOSITORY_ALREADY_PRESENT = PROJECT_ROOT.exists()
if not REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
if REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_graph_dataset.py",
    "code/python/utils/capture_graph_identity.py",
    "configs/capture_graph_stage2_input_v2.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import (
    audit_capture_graph_inputs,
    load_completed_capture_graph_input_audit,
    save_capture_graph_input_audit,
    stage_capture_graph_materialization,
)
from utils.capture_graph_identity import audit_capture_graph_identities

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Stage-2 graph-input environment is ready.")

## 2. Bind the immutable FULL_DEV materialization

The graph and prepared-packet run IDs are part of the input boundary. The audit refuses artifacts whose hashes differ from the graph-materialization provenance.

In [ ]:
FULL_DEV_RUN_ID = "20260928T175821_521784Z_graph_materialization_full_dev"
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
SOURCE_GRAPH_ROOT = (
    DRIVE_ROOT / "graph_materialization_runs" / FULL_DEV_RUN_ID
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)

for required_path in (SOURCE_GRAPH_ROOT, PREPARED_RUN_DIR, INPUT_CONTRACT_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("Materialization source:", SOURCE_GRAPH_ROOT)
print("Prepared packet source:", PREPARED_RUN_DIR)
print("Input contract:", INPUT_CONTRACT_PATH)

## 3. Stage and verify the graph collection

The collection is copied to `/content` so repeated graph access does not use Drive. Every shard and supporting artifact is checksum-verified after copying. A verified local copy is reused after rerunning the cell in the same runtime.

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=SOURCE_GRAPH_ROOT,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=INPUT_CONTRACT_PATH,
    expected_run_id=FULL_DEV_RUN_ID,
)
print("Verified local graph root:", collection.root)
print("Feature dimension:", collection.feature_dim)
print("Window duration (ms):", collection.window_ms)
print("All artifact checksums verified:",
      collection.artifact_checksums_verified)

## 4. Review the fold-role plan

A scenario has one training role and one OOF validation role across the two folds. Physical copies under both preprocessors must never be mixed inside one fold.

In [ ]:
role_rows = []
for scenario in collection.scenarios:
    role_rows.append({
        "scenario": scenario,
        "fold_A": collection.partition_for("A", scenario),
        "fold_B": collection.partition_for("B", scenario),
        "OOF_fold": next(
            fold for fold in ("A", "B")
            if collection.partition_for(fold, scenario) == "validation"
        ),
    })
role_table = pd.DataFrame(role_rows).set_index("scenario")
display(role_table)

## 5. Reconstruct a bounded sample

This reconstructs the first, middle, and last graph of every fold/scenario sequence. It checks tensor types and shapes, local node indexes, global node IDs, feature finiteness, binary targets, five-second timing, and the declared partition.

In [ ]:
sample_report = audit_capture_graph_inputs(collection, mode="SAMPLE")
sample_rows = []
for fold, scenarios in sample_report["scenario_results"].items():
    for scenario, result in scenarios.items():
        for sample in result["sampled_graphs"]:
            sample_rows.append({
                "fold": fold,
                "scenario": scenario,
                "partition": result["partition"],
                **sample,
            })
display(pd.DataFrame(sample_rows))
print("Training edges across folds:",
      sample_report["training_edges_across_folds"])
print("OOF edges across folds:", sample_report["oof_edges_across_folds"])
print("Unique development packets:",
      sample_report["unique_development_packets"])
print("OOF scenario assignment:",
      json.dumps(sample_report["oof_scenario_fold"], indent=2))

## 6. Manual approval for the full loader scan

Approve only if the role table and 30 sampled graphs are plausible, the three packet totals are equal, and all artifact checksums were verified. The full scan reconstructs every graph representation but still performs no model training.

In [ ]:
APPROVE_FULL_INPUT_AUDIT = False
SAMPLE_REVIEW_NOTE = ""

if not APPROVE_FULL_INPUT_AUDIT:
    raise RuntimeError(
        "Review the sample, then set APPROVE_FULL_INPUT_AUDIT=True."
    )
if not SAMPLE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SAMPLE_REVIEW_NOTE.")
if not collection.artifact_checksums_verified:
    raise RuntimeError("Every staged artifact checksum must be verified.")
print(SAMPLE_REVIEW_NOTE)

## 7. Run or resume the full graph-input audit

The full audit verifies contiguous `source_row_id` values and strictly increasing windows and timestamps for every graph sequence. It then joins every graph edge to the exact prepared packet, reconstructs a bijective global-node-ID-to-MAC mapping, and checks its semantic hash against both folds. The diagnostic MAC lookups are stored under the audit run and are never model inputs. Preserve the printed audit run ID.

In [ ]:
FULL_INPUT_AUDIT_RUN_ID = None  # Paste an existing completed ID to reuse it.
if FULL_INPUT_AUDIT_RUN_ID is None:
    FULL_INPUT_AUDIT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_input_full"
    )
FULL_INPUT_AUDIT_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / FULL_INPUT_AUDIT_RUN_ID
)
print("FULL input-audit run ID:", FULL_INPUT_AUDIT_RUN_ID)
print("FULL input-audit output:", FULL_INPUT_AUDIT_OUTPUT_DIR)

if FULL_INPUT_AUDIT_OUTPUT_DIR.exists():
    full_report = load_completed_capture_graph_input_audit(
        FULL_INPUT_AUDIT_OUTPUT_DIR, collection, mode="FULL"
    )
    print("Reused the completed immutable input audit.")
else:
    full_report = audit_capture_graph_inputs(collection, mode="FULL")
    identity_result = audit_capture_graph_identities(
        collection,
        prepared_run_dir=PREPARED_RUN_DIR,
        local_work_root=LOCAL_IDENTITY_WORK_ROOT,
    )
    full_report["identity_audit"] = identity_result["report"]
    full_report["sample_review_note"] = SAMPLE_REVIEW_NOTE
    save_capture_graph_input_audit(
        FULL_INPUT_AUDIT_OUTPUT_DIR,
        full_report,
        contract_path=INPUT_CONTRACT_PATH,
        identity_lookups=identity_result["lookups"],
    )
    print("Saved the completed immutable input audit.")

## 8. Review the full input boundary

Every row must report a complete sequence with contiguous source rows and increasing windows/timestamps. Every identity row must report a conflict-free bijection whose reconstructed semantic hash matches both folds. Passing this gate authorizes implementation of the frozen one-seed training runner; it is not model evidence.

In [ ]:
full_rows = []
for fold, scenarios in full_report["scenario_results"].items():
    for scenario, result in scenarios.items():
        full_rows.append(result)
full_table = pd.DataFrame(full_rows)
display(full_table)
identity_rows = []
for scenario, result in full_report["identity_audit"]["scenarios"].items():
    identity_rows.append({"scenario": scenario, **result})
identity_table = pd.DataFrame(identity_rows)
display(identity_table)
print("Training edges across folds:",
      full_report["training_edges_across_folds"])
print("OOF edges across folds:", full_report["oof_edges_across_folds"])
print("Unique development packets:",
      full_report["unique_development_packets"])
print("Held-out scenarios accessed:",
      full_report["held_out_scenarios_accessed"])
print("Training performed:", full_report["training_performed"])
print("Audit report:",
      FULL_INPUT_AUDIT_OUTPUT_DIR / "capture_graph_input_audit.json")
print("Diagnostic identity lookups:",
      FULL_INPUT_AUDIT_OUTPUT_DIR / "node_identity_lookup")

required_flags = [
    "complete_sequence",
    "source_rows_contiguous",
    "windows_and_timestamps_strictly_increasing",
]
if not full_table[required_flags].all(axis=None):
    raise RuntimeError("The full graph-input boundary audit failed.")
identity_required = [
    "source_rows_contiguous",
    "first_seen_rows_match_stored_mapping",
    "reconstructed_contract_matches_both_folds",
]
if not identity_table[identity_required].all(axis=None):
    raise RuntimeError("The node-identity boundary audit failed.")
if not (
    identity_table["id_to_mac_conflicts"].eq(0).all()
    and identity_table["mac_to_id_conflicts"].eq(0).all()
    and identity_table["unresolved_global_nodes"].eq(0).all()
):
    raise RuntimeError("The recovered node mapping is not bijective.")
if not (
    full_report["training_edges_across_folds"]
    == full_report["oof_edges_across_folds"]
    == full_report["unique_development_packets"]
):
    raise RuntimeError("Fold roles do not cover each packet exactly once.")